# Voice Intent Classifier — Atlas Ledger Wave 13C

Fine-tunes DistilBERT for 10-class intent classification from voice/text utterances.
Exports INT8 ONNX for on-device inference via onnxruntime-react-native.

| Intent | ID | Example |
|--------|-----|---------|
| ADD_EXPENSE | 0 | "add $45 at Starbucks" |
| QUERY_SPENDING | 1 | "how much did I spend this month" |
| QUERY_BUDGET | 2 | "am I on track with my budget" |
| QUERY_CATEGORY | 3 | "how much on food this month" |
| QUERY_MERCHANT | 4 | "how much at Costco" |
| QUERY_DATE_RANGE | 5 | "expenses last quarter" |
| SET_BUDGET | 6 | "set food budget to 500" |
| LOG_MILEAGE | 7 | "log 45 miles" |
| EXPORT_DATA | 8 | "export my expenses" |
| OTHER | 9 | "never mind" |

In [ ]:
!pip install -q transformers torch scikit-learn onnxruntime accelerate onnx onnxscript


In [ ]:
import random, json, os, numpy as np, torch
from pathlib import Path
random.seed(42); np.random.seed(42); torch.manual_seed(42)

INTENT_MAP = {0:"ADD_EXPENSE",1:"QUERY_SPENDING",2:"QUERY_BUDGET",3:"QUERY_CATEGORY",
              4:"QUERY_MERCHANT",5:"QUERY_DATE_RANGE",6:"SET_BUDGET",7:"LOG_MILEAGE",
              8:"EXPORT_DATA",9:"OTHER"}
MODEL_CHECKPOINT = "distilbert-base-uncased"

In [ ]:
MERCHANTS = ["Starbucks","Whole Foods","Shell","Costco","Amazon","Target","Uber","McDonald's","CVS","Home Depot"]
AMOUNTS = ["$5","$12","$18","$25","$45","$67","$100","$150","$200","$350","5 dollars","12 bucks","45 dollars"]
PERIODS = ["today","this week","this month","last month","last week","this year","last quarter","in November","in Q3","yesterday"]
CATEGORIES = ["food","dining","groceries","gas","coffee","shopping","entertainment","travel","medical","office supplies"]

templates = {
  0: ["add {a} at {m}", "spent {a} on {c}", "log {a} {c} {p}", "record {a} at {m}", "add expense {a} for {c}",
      "I spent {a} at {m}", "charge {a} to {c}", "track {a} at {m} {p}", "new expense {a} {m}", "put {a} for {c}"],
  1: ["how much did I spend {p}", "what's my total spending {p}", "show me my expenses {p}",
      "total expenses {p}", "how much have I spent {p}", "spending summary {p}", "what did I spend {p}",
      "my expenses {p}", "spending total {p}", "expense report {p}"],
  2: ["am I on track with my budget", "how much budget do I have left", "budget status",
      "how am I doing on budget", "budget remaining", "am I over budget", "budget check",
      "how's my budget looking", "budget update", "am I within budget"],
  3: ["how much on {c} {p}", "show {c} expenses", "{c} spending {p}", "how much for {c}",
      "expenses in {c}", "total {c} {p}", "{c} budget status", "how much did I spend on {c}",
      "show me {c} costs", "{c} charges {p}"],
  4: ["how much at {m}", "{m} expenses {p}", "how much did I spend at {m}", "charges from {m}",
      "{m} total {p}", "show {m} purchases", "what did I spend at {m}", "{m} spending",
      "how often do I go to {m}", "{m} charges this month"],
  5: ["expenses last quarter", "show me {p} spending", "what did I spend {p}", "expenses for {p}",
      "spending in {p}", "report for {p}", "charges {p}", "what were my expenses {p}",
      "total for {p}", "breakdown for {p}"],
  6: ["set {c} budget to {a}", "update {c} limit to {a}", "change budget for {c} to {a}",
      "budget {c} at {a}", "set spending limit for {c}", "update my {c} budget",
      "new budget for {c}", "I want to spend {a} on {c}", "limit {c} to {a}", "budget goal for {c}"],
  7: ["log {n} miles", "record {n} mile trip", "add {n} miles to client",
      "I drove {n} miles", "mileage {n} miles", "track {n} miles",
      "log trip {n} miles", "add mileage {n}", "drove {n} miles today", "record mileage {n} miles"],
  8: ["export my expenses", "download CSV", "export to spreadsheet", "share expense report",
      "export data", "generate report", "download my expenses", "export as Excel",
      "send expense report", "export {p} expenses"],
  9: ["hello", "thanks", "never mind", "cancel", "go back", "help",
      "what can you do", "stop", "exit", "ok"]
}

def fill(t, **kw):
    try:
        return t.format(**kw)
    except KeyError:
        return t

data = []
for intent_id, tmpl_list in templates.items():
    for _ in range(20):  # 20 rounds x 10 templates = 200 per class
        for t in tmpl_list:
            n = random.randint(5, 250)
            text = fill(t, m=random.choice(MERCHANTS), a=random.choice(AMOUNTS),
                        p=random.choice(PERIODS), c=random.choice(CATEGORIES), n=n)
            data.append({"text": text, "label": intent_id})

random.shuffle(data)
n = len(data); n_train = int(n*0.8); n_val = int(n*0.1)
train_data = data[:n_train]
val_data = data[n_train:n_train+n_val]
test_data = data[n_train+n_val:]
print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

In [ ]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

class IntentDataset(Dataset):
    def __init__(self, items):
        self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        enc = tokenizer(self.items[i]["text"], truncation=True, max_length=64, padding="max_length", return_tensors="pt")
        return {k: v.squeeze(0) for k, v in enc.items()} | {"labels": torch.tensor(self.items[i]["label"])}

train_ds = IntentDataset(train_data)
val_ds = IntentDataset(val_data)
test_ds = IntentDataset(test_data)
print("Datasets ready")

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, accuracy_score

model = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT, num_labels=10)

def compute_metrics(p):
    preds = p.predictions.argmax(-1)
    return {"accuracy": accuracy_score(p.label_ids, preds),
            "f1_macro": f1_score(p.label_ids, preds, average="macro")}

args = TrainingArguments(
    output_dir="./voice-intent-output",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  compute_metrics=compute_metrics)
trainer.train()
print("Training complete")

In [ ]:
from sklearn.metrics import classification_report

preds_out = trainer.predict(test_ds)
preds = preds_out.predictions.argmax(-1)
labels = preds_out.label_ids

report = classification_report(labels, preds, target_names=[INTENT_MAP[i] for i in range(10)], output_dict=True)
print(classification_report(labels, preds, target_names=[INTENT_MAP[i] for i in range(10)]))

THRESHOLDS = {
    "ADD_EXPENSE": 0.90, "QUERY_SPENDING": 0.85, "QUERY_BUDGET": 0.85,
    "QUERY_CATEGORY": 0.85, "QUERY_MERCHANT": 0.85, "QUERY_DATE_RANGE": 0.85,
    "SET_BUDGET": 0.80, "LOG_MILEAGE": 0.80, "EXPORT_DATA": 0.80, "OTHER": 0.75,
}

print("\n--- Acceptance Gate ---")
all_pass = True
for intent, threshold in THRESHOLDS.items():
    f1 = report[intent]["f1-score"]
    status = "\u2705" if f1 >= threshold else "\u274c"
    print(f"{status} {intent}: F1={f1:.3f} (min {threshold})")
    if f1 < threshold:
        all_pass = False

if all_pass:
    print("\n\u2705 All thresholds met \u2014 ready to export")
else:
    print("\n\u26a0\ufe0f  Some thresholds not met. Consider more epochs or data augmentation.")

In [ ]:
import torch, os, onnx
from onnxruntime.quantization import quantize_dynamic, QuantType

os.makedirs("release", exist_ok=True)
model.cpu().eval()

dummy_input_ids = torch.zeros(1, 64, dtype=torch.long)
dummy_attention_mask = torch.ones(1, 64, dtype=torch.long)

# Verify output shape
with torch.no_grad():
    out = model(input_ids=dummy_input_ids, attention_mask=dummy_attention_mask)
    print("Logits shape:", out.logits.shape)  # must be [1, 10]

torch.onnx.export(
    model,
    (dummy_input_ids, dummy_attention_mask),
    "release/voice-intent-fp32.onnx",
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={"input_ids": {0: "batch"}, "attention_mask": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=14,
    do_constant_folding=False,
)
print("FP32 export complete:", round(os.path.getsize("release/voice-intent-fp32.onnx")/1e6, 1), "MB")

# Strip intermediate value_info shape annotations — they conflict with onnx.shape_inference
# inside quantize_dynamic (known issue with HuggingFace classifier head: 768 vs 10).
# Clearing value_info lets shape inference run from scratch without conflicts.
m = onnx.load("release/voice-intent-fp32.onnx")
del m.graph.value_info[:]
onnx.save(m, "release/voice-intent-fp32-stripped.onnx")
print("Shape annotations stripped")

quantize_dynamic(
    "release/voice-intent-fp32-stripped.onnx",
    "release/voice-intent-classifier.onnx",
    weight_type=QuantType.QInt8,
)
print("INT8 export complete:", round(os.path.getsize("release/voice-intent-classifier.onnx")/1e6, 1), "MB")


In [ ]:
import onnxruntime as ort, json, os

# Validate ONNX
sess = ort.InferenceSession("release/voice-intent-classifier.onnx")
enc = tokenizer("how much did I spend this month", truncation=True, max_length=64, padding="max_length", return_tensors="np")
logits = sess.run(["logits"], {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"]})[0]
pred = int(logits.argmax(-1)[0])
print(f"Sample inference: '{INTENT_MAP[pred]}' (expected: QUERY_SPENDING)")

# Save intent_map.json
with open("release/intent_map.json", "w") as f:
    json.dump({str(k): v for k, v in INTENT_MAP.items()}, f, indent=2)

# Save vocab — save_pretrained writes vocab.txt; save_vocabulary returns paths to saved files
saved = tokenizer.save_pretrained("release/")
print("Tokenizer saved:", [os.path.basename(p) for p in saved if p])

print("Release files ready:", os.listdir("release/"))


## Creating the GitHub Release

```bash
gh release create voice-intent-v1.0 \
  release/voice-intent-classifier.onnx \
  release/vocab.txt \
  release/intent_map.json \
  --repo buzybee83/atlas-ledger-models \
  --title "Voice Intent v1.0" \
  --notes "DistilBERT INT8 ONNX. 10-class intent classifier for voice/text. F1 macro >= 0.85."
```

Then set `VOICE_INTENT_MODEL_DOWNLOAD_URL` in `src/services/VoiceIntentService.ts`:

```typescript
const VOICE_INTENT_MODEL_DOWNLOAD_URL =
  'https://github.com/buzybee83/atlas-ledger-models/releases/download/voice-intent-v1.0/voice-intent-classifier.onnx';
```

In [ ]:
import os

try:
    from google.colab import files

    files.download("release/voice-intent-classifier.onnx")
    files.download("release/intent_map.json")

    # Download whichever vocab file the tokenizer saved
    for name in ["vocab.txt", "tokenizer.json", "tokenizer_config.json"]:
        path = f"release/{name}"
        if os.path.exists(path):
            files.download(path)
            print(f"Downloaded: {name}")

    print("Downloads started")
except ImportError:
    print("Not in Colab — files are in release/ directory")
    print("Contents:", os.listdir("release/"))
